In [ ]:
!pip install -q torch-geometric
import torch
from torch_geometric.datasets import MD17
import time
import os

print("Starting FoldPipe Data Serialization...")
start_time = time.time()

# 1. Load the original, unoptimized dataset
dataset = MD17(root='./data', name='aspirin')
num_graphs = len(dataset)
print(f"Loaded {num_graphs} original complex graph objects.")

# 2. Pre-allocate massive, flat memory blocks (contiguous tensors)
# Aspirin has 21 atoms. We pre-build the exact matrix shapes.
num_atoms = dataset[0].z.shape[0]

print("\nFlattening data architecture...")
# We only need one copy of the atomic numbers (they don't change)
z_tensor = dataset[0].z 
# Allocate contiguous blocks for positions, energies, and forces
pos_tensor = torch.zeros((num_graphs, num_atoms, 3), dtype=torch.float32)
energy_tensor = torch.zeros((num_graphs, 1), dtype=torch.float32)
force_tensor = torch.zeros((num_graphs, num_atoms, 3), dtype=torch.float32)

# 3. Extract and pack the data (This strips away all Python object overhead)
for i in range(num_graphs):
    data = dataset[i]
    pos_tensor[i] = data.pos
    energy_tensor[i] = data.energy
    force_tensor[i] = data.force

# 4. Save to a contiguous binary file on disk
os.makedirs('./foldpipe_data', exist_ok=True)
torch.save({
    'z': z_tensor,
    'pos': pos_tensor,
    'energy': energy_tensor,
    'force': force_tensor
}, './foldpipe_data/aspirin_flat.pt')

end_time = time.time()

print("\n" + "="*50)
print("SERIALIZATION COMPLETE")
print("="*50)
print(f"Converted {num_graphs} graphs into flat binary blocks.")
print(f"Time taken: {end_time - start_time:.2f} seconds.")
print("Output saved to: ./foldpipe_data/aspirin_flat.pt")
print("="*50)
print("The data is now optimized. We are ready to build the asynchronous loader.")
